In [ ]:
library(here)
library(maplet)
library(dplyr)
library(purrr)

# set repo path
repo <- here()
renv::activate(project = repo)

In [ ]:
# load maplet object
D <- readRDS(here('data', 'preprocessed_venous_metabolon.RDS'))

# Define PVR Status

In [ ]:
# define a 'PVR_Status' variable
D <- D %>% 
    mt_anno_mutate(anno_type="samples", col_name="PVR_status", term = 
        case_when(
            pvr_spont <= 2 ~ 'PVR_low', 
            pvr_spont > 2 & pvr_spont <= 5 ~ 'PVR_else',
            pvr_spont >= 5 ~ 'PVR_high',
            is.na(pvr_spont) ~ NA_character_,
            TRUE ~ 'Undefined'
        )
    )

# Moderation Effect Analysis (PVR-High/PVR-Low)

In [ ]:
# Define confounders as a string
confounders <- "age + SEX + bmi"

# isolate metabolite names
met_names <- rownames(D)

First, we'll subset our maplet object to isolate RV-specific metabolite profiles that are non-missing. 

In [ ]:
D1_GLS <- D %>% 
    # exclude non-WSPH groups
    mt_modify_filter_samples(filter = !is.na(pvr_spont)) %>% 
    # filter out samples with missing RVGLOB6n parameters
    mt_modify_filter_samples(filter = !is.na(RVGLOB6n))

In [ ]:
D1_FAC <- D %>% 
    # exclude non-WSPH groups
    mt_modify_filter_samples(filter = !is.na(pvr_spont)) %>% 
    # filter out samples with missing RVFACn parameters
    mt_modify_filter_samples(filter = !is.na(RVFACn))

In [ ]:
D1_RVEF <- D %>% 
    # exclude non-WSPH groups
    mt_modify_filter_samples(filter = !is.na(pvr_spont)) %>% 
    # filter out samples with missing mri_RVEF parameters
    mt_modify_filter_samples(filter = !is.na(mri_RVEF))

In [ ]:
# function to prepare a df of who_groups, mets, and clin_var
interaction_df_prep <- function(D, clin_var) { 
    clin_df <- D %>% colData() %>% as.data.frame() %>% 
        # select the clinical variable
        select(!!sym(clin_var), pvr_spont, age, SEX, bmi) %>% tibble::rownames_to_column('StudyID') %>% na.omit()

    assay_df <- D %>% assay() %>% t() %>% as.data.frame() %>% tibble::rownames_to_column('StudyID')

    # join these by StudyID
    joined_df <- clin_df %>% inner_join(assay_df, by = 'StudyID') %>% 
        tibble::column_to_rownames('StudyID')
}

In [ ]:
library(broom)
library(dplyr)
library(stringr)
library(purrr)

run_moderation_models <- function(data, clin_var, met_names, moderator_var = "pvr_spont", covars = NULL) {
  
  results_list <- list()
  
  for (met in met_names) {
    
    # Create the formula: outcome ~ clinical * moderator + covariates
    interaction_term <- paste0(clin_var, "*", moderator_var)
    covar_str <- if (!is.null(covars) && length(covars) > 0) paste(covars, collapse = " + ") else NULL
    
    rhs <- paste(c(interaction_term, covar_str), collapse = " + ")
    formula_str <- paste0("`", met, "` ~ ", rhs)
    fmla <- as.formula(formula_str)
    
    # Fit the model
    model <- lm(fmla, data = data)
    
    # Tidy the results
    tidy_results <- broom::tidy(model)
    
    # Add metadata
    tidy_results <- tidy_results %>%
      mutate(
        met_name = met,
        
        # Coarse label
        coefficient_type = case_when(
          term == "(Intercept)" ~ "Intercept",
          term == clin_var ~ "ClinicalVar",
          term == moderator_var ~ "Moderator",
          str_detect(term, ":") ~ "Interaction",
          TRUE ~ "Other"
        ),
        
        # Fine-grained label
        coefficient_type_2 = case_when(
          term == "(Intercept)" ~ "Intercept",
          term == clin_var ~ "ClinicalVar",
          term == moderator_var ~ "Moderator",
          str_detect(term, paste0(clin_var, ":", moderator_var)) |
            str_detect(term, paste0(moderator_var, ":", clin_var)) ~ "Interaction",
          TRUE ~ paste0("Other_", term)
        )
      )
    
    # Store in list
    results_list[[met]] <- tidy_results
  }
  
  # Combine all results
  results_df <- bind_rows(results_list)
  
  return(results_df)
}


## GLS 6`

In [ ]:
var_of_interest <- "RVGLOB6n"

# prepare df to interaction term anaylsis
interaction_df <- interaction_df_prep(D1_GLS, clin_var = var_of_interest)

results_df <- run_moderation_models(
  data = interaction_df, #  df containing pvr_spont, metabolite profiles, and clinical confounders
  clin_var = var_of_interest, # rv parameter
  met_names = met_names, # metabolite names
  moderator_var = "pvr_spont", # continuous pvr variable
  covars = confounders # confounders
)

In [ ]:
# prepare table for export (main effect and interaction term)
stats_main <- results_df %>% filter(coefficient_type == 'ClinicalVar') %>% mutate(FDR = p.adjust(p.value, method = 'fdr')) 
stats_interaction <- results_df %>% filter(coefficient_type == 'Interaction') %>% mutate(FDR = p.adjust(p.value, method = 'fdr'))

# export
write.csv(stats_main, file.path(here('outputs', paste0(var_of_interest, "_pvr_moderation_effect_analysis_main.xlsx"))))
write.csv(stats_interaction, file.path(here('outputs', paste0(var_of_interest, "_pvr_moderation_effect_analysis_interaction.xlsx"))))

## FAC

In [ ]:
var_of_interest <- "RVFACn"

# prepare df to interaction term anaylsis
interaction_df <- interaction_df_prep(D1_FAC, clin_var = var_of_interest)

results_df <- run_moderation_models(
  data = interaction_df, #  df containing pvr_spont, metabolite profiles, and clinical confounders
  clin_var = var_of_interest, # rv parameter
  met_names = met_names, # metabolite names
  moderator_var = "pvr_spont", # continuous pvr variable
  covars = confounders # confounders
)

In [ ]:
# prepare table for export (main effect and interaction term)
stats_main <- results_df %>% filter(coefficient_type == 'ClinicalVar') %>% mutate(FDR = p.adjust(p.value, method = 'fdr')) 
stats_interaction <- results_df %>% filter(coefficient_type == 'Interaction') %>% mutate(FDR = p.adjust(p.value, method = 'fdr'))

# export
write.csv(stats_main, file.path(here('outputs', paste0(var_of_interest, "_pvr_moderation_effect_analysis_main.xlsx"))))
write.csv(stats_interaction, file.path(here('outputs', paste0(var_of_interest, "_pvr_moderation_effect_analysis_interaction.xlsx"))))

## RVEF

In [ ]:
var_of_interest <- "mri_RVEF"

# prepare df to interaction term anaylsis
interaction_df <- interaction_df_prep(D1_RVEF, clin_var = var_of_interest)

results_df <- run_moderation_models(
  data = interaction_df, #  df containing pvr_spont, metabolite profiles, and clinical confounders
  clin_var = var_of_interest, # rv parameter
  met_names = met_names, # metabolite names
  moderator_var = "pvr_spont", # continuous pvr variable
  covars = confounders # confounders
)

In [ ]:
# prepare table for export (main effect and interaction term)
stats_main <- results_df %>% filter(coefficient_type == 'ClinicalVar') %>% mutate(FDR = p.adjust(p.value, method = 'fdr')) 
stats_interaction <- results_df %>% filter(coefficient_type == 'Interaction') %>% mutate(FDR = p.adjust(p.value, method = 'fdr'))

# export
write.csv(stats_main, file.path(here('outputs', paste0(var_of_interest, "_pvr_moderation_effect_analysis_main.xlsx"))))
write.csv(stats_interaction, file.path(here('outputs', paste0(var_of_interest, "_pvr_moderation_effect_analysis_interaction.xlsx"))))